In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install pyotp mplfinance pandas_ta fyers_apiv3 pygame stable-baselines3 gymnasium shimmy optuna higher torch_xla

In [3]:
import requests
import base64
from datetime import datetime, timedelta, date
from datetime import time as dt_time
import time
import threading
import pyotp
from pytz import timezone
import pandas as pd
import numpy as np
from numba import njit, prange
from urllib.parse import urlparse, parse_qs
import matplotlib.pyplot as plt
import mplfinance as mpf
import pandas_ta as ta
import pygame
import pytz
import os
import json
import sys
import gc

import gymnasium as gym
from gymnasium import spaces
from IPython.display import display, clear_output
from tqdm import tqdm

from fyers_apiv3 import fyersModel
from fyers_apiv3.FyersWebsocket import data_ws

from scipy.signal import argrelextrema
import tensorflow as tf
from collections import deque
from sklearn.preprocessing import StandardScaler

pygame 2.6.1 (SDL 2.28.4, Python 3.11.11)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [4]:
app_id = "TS79V3NXK1-100"
secret_key = "KQCPB0FJ74"
redirect_uri = "https://google.com"
fyers_user = "XM22383"
fyers_pin = "4628"
fyers_totp = "EAQD6K4IUYOEGPJNVE6BMPTUSDCWIOHW"
response_type = "code"
state = "sample_state"
grant_type = "authorization_code"

fyers = None
fyers_socket = None

ist_timezone = pytz.timezone("Asia/Kolkata")

# Initialize the mixer
#pygame.mixer.init()

# Load the sound file
#pygame.mixer.music.load('sounds/alert.mp3')

#Variables
ce_ltp = 0
pe_ltp = 0
index_ltp = 0
buy_sell_checked = False
ce_strike = None
pe_strike = None
ce_symbol = None
pe_symbol = None

fixed_ltp = 0
fixed_index_ltp = 0
prev_ltp = 0
target_inside = 0
target_index_inside = 0
trailing_sl_inside = 0
trailing_index_inside = 0

active_order = False

sl_hit_condition = False
total_loss = 0
total_profit = 0
overall_win = 0
overall_loss = 0
total_points = 0

unsubscribe_done = False

active_order_sleep = 1

In [5]:
session = fyersModel.SessionModel(
    client_id=app_id,
    secret_key=secret_key,
    redirect_uri=redirect_uri,
    response_type=response_type,
    grant_type=grant_type
)

def getEncodedString(string):
    string = str(string)
    base64_bytes = base64.b64encode(string.encode("ascii"))
    return base64_bytes.decode("ascii")

if session is not None:
    session.generate_authcode()

    url_send_login_otp = "https://api-t2.fyers.in/vagator/v2/send_login_otp_v2"
    res = requests.post(url=url_send_login_otp, json={"fy_id": getEncodedString(fyers_user), "app_id": "2"}).json()

    if datetime.now().second % 30 > 27:
        time.sleep(5)

    url_verify_otp = "https://api-t2.fyers.in/vagator/v2/verify_otp"
    res2 = requests.post(url=url_verify_otp, json={"request_key": res["request_key"], "otp": pyotp.TOTP(fyers_totp).now()}).json()

    ses = requests.Session()
    url_verify_otp2 = "https://api-t2.fyers.in/vagator/v2/verify_pin_v2"
    payload2 = {"request_key": res2["request_key"], "identity_type": "pin", "identifier": getEncodedString(fyers_pin)}
    res3 = ses.post(url=url_verify_otp2, json=payload2).json()

    ses.headers.update({'authorization': f"Bearer {res3['data']['access_token']}"})

    tokenurl = "https://api-t1.fyers.in/api/v3/token"
    payload3 = {
        "fyers_id": fyers_user,
        "app_id": app_id[:-4],
        "redirect_uri": redirect_uri,
        "appType": "100",
        "code_challenge": "",
        "state": "None",
        "scope": "",
        "nonce": "",
        "response_type": "code",
        "create_cookie": True
    }

    res3 = ses.post(url=tokenurl, json=payload3).json()

    url = res3['Url']
    parsed = urlparse(url)
    auth_code = parse_qs(parsed.query)['auth_code'][0]

    session.set_token(auth_code)

    auth_response = session.generate_token()
    access_token = auth_response["access_token"]

    fyers = fyersModel.FyersModel(client_id=app_id, token=access_token)

    ws_token = app_id + ":" + access_token
    fyers_socket = data_ws.FyersDataSocket(access_token=ws_token, log_path="")

pd.DataFrame(fyers.get_profile())

,s,code,message,data
fy_id,ok,200,,XM22383
name,ok,200,,MARSHAL TUDU
image,ok,200,,https://myaccount-docs-prod.fyers.in/Profile_P...
display_name,ok,200,,None
pin_change_date,ok,200,,25-09-2023 17:16:16
email_id,ok,200,,iammarshal22@gmail.com
pwd_change_date,ok,200,,01-06-2022 20:36:31
PAN,ok,200,,---------
mobile_number,ok,200,,8458060663
totp,ok,200,,True


In [6]:
def fetch_candle_data(number, index_symbol, interval_minutes):
    while True:
        try:
            today = date.today()
            yesterday = today - timedelta(number)

            data = {
                "symbol": index_symbol,
                "resolution": interval_minutes,
                "date_format": "1",
                "range_from": yesterday,
                "range_to": today,
                "cont_flag": "1"
            }

            result = fyers.history(data=data)

            if result is not None:
                train_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                return train_df
        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [7]:
def fetch_train_candle_data(days_count, index_symbol, interval_minutes):
    train_df = pd.DataFrame()

    while True:
        try:
            date_increment = 100
            for i in range(days_count):
                today = date.today() - timedelta(date_increment)
                yesterday = today - timedelta(100)

                data = {
                    "symbol": index_symbol,
                    "resolution": interval_minutes,
                    "date_format": "1",
                    "range_from": yesterday,
                    "range_to": today,
                    "cont_flag": "1"
                }

                result = fyers.history(data=data)

                if result is not None:
                    temp_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                    train_df = pd.concat([temp_df, train_df], ignore_index=True)

                date_increment += 100

            if train_df is not None:
                return train_df

        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [8]:
index_mapping = {
    # Indices
    "Nifty": {
        "symbol": "NSE:NIFTY50-INDEX",
        "quantity": 75
    },
    "Bank_Nifty": {
        "symbol": "NSE:NIFTYBANK-INDEX",
        "quantity": 30
    },
    "Finnifty": {
        "symbol": "NSE:FINNIFTY-INDEX",
        "quantity": 40
    },
    "Sensex": {
        "symbol": "BSE:SENSEX-INDEX",
        "quantity": 20
    },
    "Bankex": {
        "symbol": "BSE:BANKEX-INDEX",
        "quantity": 15
    },

    # Complete Nifty 50 Stocks

    "Adani_Enterprises_Ltd": {
        "symbol": "NSE:ADANIENT-EQ",
        "quantity": 10
    },
    "Adani_Ports__SEZ_Ltd": {
        "symbol": "NSE:ADANIPORTS-EQ",
        "quantity": 10
    },
    "Apollo_Hospitals_Enterprise_Ltd": {
        "symbol": "NSE:APOLLOHOSP-EQ",
        "quantity": 10
    },
    "Asian_Paints_Ltd": {
        "symbol": "NSE:ASIANPAINT-EQ",
        "quantity": 10
    },
    "Axis_Bank_Ltd": {
        "symbol": "NSE:AXISBANK-EQ",
        "quantity": 10
    },
    "Bajaj_Auto_Ltd": {
        "symbol": "NSE:BAJAJ-AUTO-EQ",
        "quantity": 10
    },
    "Bajaj_Finance_Ltd": {
        "symbol": "NSE:BAJFINANCE-EQ",
        "quantity": 10
    },
    "Bajaj_Finserv_Ltd": {
        "symbol": "NSE:BAJAJFINSV-EQ",
        "quantity": 10
    },
    "Bharat_Electronics_Ltd": {
        "symbol": "NSE:BEL-EQ",
        "quantity": 10
    },
    "Bharti_Airtel_Ltd": {
        "symbol": "NSE:BHARTIARTL-EQ",
        "quantity": 10
    },
    "BPCL_Ltd": {
        "symbol": "NSE:BPCL-EQ",
        "quantity": 10
    },
    "Cipla_Ltd": {
        "symbol": "NSE:CIPLA-EQ",
        "quantity": 10
    },
    "Coal_India_Ltd": {
        "symbol": "NSE:COALINDIA-EQ",
        "quantity": 10
    },
    "Dr_Reddys_Laboratories_Ltd": {
        "symbol": "NSE:DRREDDY-EQ",
        "quantity": 10
    },
    "Eicher_Motors_Ltd": {
        "symbol": "NSE:EICHERMOT-EQ",
        "quantity": 10
    },
    "Grasim_Industries_Ltd": {
        "symbol": "NSE:GRASIM-EQ",
        "quantity": 10
    },
    "HCL_Technologies_Ltd": {
        "symbol": "NSE:HCLTECH-EQ",
        "quantity": 10
    },
    "HDFC_Bank_Ltd": {
        "symbol": "NSE:HDFCBANK-EQ",
        "quantity": 10
    },
    "HDFC_Life_Insurance_Company_Ltd": {
        "symbol": "NSE:HDFCLIFE-EQ",
        "quantity": 10
    },
    "Hero_MotoCorp_Ltd": {
        "symbol": "NSE:HEROMOTOCO-EQ",
        "quantity": 10
    },
    "Hindalco_Industries_Ltd": {
        "symbol": "NSE:HINDALCO-EQ",
        "quantity": 10
    },
    "Hindustan_Unilever_Ltd": {
        "symbol": "NSE:HINDUNILVR-EQ",
        "quantity": 10
    },
    "ICICI_Bank_Ltd": {
        "symbol": "NSE:ICICIBANK-EQ",
        "quantity": 10
    },
    "IndusInd_Bank_Ltd": {
        "symbol": "NSE:INDUSINDBK-EQ",
        "quantity": 10
    },
    "Infosys_Ltd": {
        "symbol": "NSE:INFY-EQ",
        "quantity": 10
    },
    "Indian_Oil_Corporation_Ltd": {
        "symbol": "NSE:IOC-EQ",
        "quantity": 10
    },
    "JSW_Steel_Ltd": {
        "symbol": "NSE:JSWSTEEL-EQ",
        "quantity": 10
    },
    "Kotak_Mahindra_Bank_Ltd": {
        "symbol": "NSE:KOTAKBANK-EQ",
        "quantity": 10
    },
    "Larsen__Toubro_Ltd": {
        "symbol": "NSE:LT-EQ",
        "quantity": 10
    },
    "Mahindra__Mahindra_Ltd": {
        "symbol": "NSE:M&M-EQ",
        "quantity": 10
    },
    "Maruti_Suzuki_India_Ltd": {
        "symbol": "NSE:MARUTI-EQ",
        "quantity": 10
    },
    "Nestle_India_Ltd": {
        "symbol": "NSE:NESTLEIND-EQ",
        "quantity": 10
    },
    "NTPC_Ltd": {
        "symbol": "NSE:NTPC-EQ",
        "quantity": 10
    },
    "ONGC_Ltd": {
        "symbol": "NSE:ONGC-EQ",
        "quantity": 10
    },
    "Power_Grid_Corporation_of_India_Ltd": {
        "symbol": "NSE:POWERGRID-EQ",
        "quantity": 10
    },
    "Reliance_Industries_Ltd": {
        "symbol": "NSE:RELIANCE-EQ",
        "quantity": 10
    },
    "State_Bank_of_India_(SBI)_Ltd": {
        "symbol": "NSE:SBIN-EQ",
        "quantity": 10
    },
    "SBI_Life_Insurance_Company_Ltd": {
        "symbol": "NSE:SBILIFE-EQ",
        "quantity": 10
    },
    "Sun_Pharmaceutical_Industries_Ltd": {
        "symbol": "NSE:SUNPHARMA-EQ",
        "quantity": 10
    },
    "Tata_Consumer_Products_Ltd": {
        "symbol": "NSE:TATACONSUM-EQ",
        "quantity": 10
    },
    "Tata_Motors_Ltd": {
        "symbol": "NSE:TATAMOTORS-EQ",
        "quantity": 10
    },
    "Tata_Steel_Ltd": {
        "symbol": "NSE:TATASTEEL-EQ",
        "quantity": 10
    },
    "TCS_Ltd": {
        "symbol": "NSE:TCS-EQ",
        "quantity": 10
    },
    "Tech_Mahindra_Ltd": {
        "symbol": "NSE:TECHM-EQ",
        "quantity": 10
    },
    "Titan_Company_Ltd": {
        "symbol": "NSE:TITAN-EQ",
        "quantity": 10
    },
    "Trent_Ltd": {
        "symbol": "NSE:TRENT-EQ",
        "quantity": 10
    },
    "UltraTech_Cement_Ltd": {
        "symbol": "NSE:ULTRACEMCO-EQ",
        "quantity": 10
    },
    "Wipro_Ltd": {
        "symbol": "NSE:WIPRO-EQ",
        "quantity": 10
    }
}

In [9]:
# Historical config for fetching and saving raw data
history_config = {
    "timeframes": [2, 3, 5, 10, 15, 20, 30, 45, 60, 120, 180, 240],
    "colab_folder_path": "/content/drive/MyDrive/historical_data",
    "colab_folder_path_processed": "processed_data",
    "local_folder_path": "historical_data",
    "local_folder_path_processed": "C:\\Users\\gs002\\OneDrive\\Desktop\\Marshal",
    "bar_limit": 15  # parameter for fetch_train_candle_data
}

In [10]:
if "google.colab" in sys.modules:
    folder_path = history_config["colab_folder_path"]
else:
    folder_path = history_config["local_folder_path"]

In [11]:
# Ensure the folder exists
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

In [12]:
# Main loop to fetch and save data
for instrument_name, info in index_mapping.items():
    for tf in history_config["timeframes"]:
        file_name = f"{instrument_name}_{tf}.csv"
        file_path = os.path.join(folder_path, file_name)

        # Skip if file already exists
        if os.path.exists(file_path):
            print(f"File already exists for {instrument_name} timeframe {tf}. Skipping.")
            continue

        # Fetch data for given bar_limit, symbol, and timeframe
        df = fetch_train_candle_data(history_config["bar_limit"], info["symbol"], tf)

        # Save if data is not empty
        if not df.empty:
            df.to_csv(file_path, index=False)
            print(f"Saved data for {instrument_name} timeframe {tf} at {file_path}")
        else:
            print(f"No data returned for {instrument_name} timeframe {tf}.")

File already exists for Nifty timeframe 2. Skipping.
File already exists for Nifty timeframe 3. Skipping.
File already exists for Nifty timeframe 5. Skipping.
File already exists for Nifty timeframe 10. Skipping.
File already exists for Nifty timeframe 15. Skipping.
File already exists for Nifty timeframe 20. Skipping.
File already exists for Nifty timeframe 30. Skipping.
File already exists for Nifty timeframe 45. Skipping.
File already exists for Nifty timeframe 60. Skipping.
File already exists for Nifty timeframe 120. Skipping.
File already exists for Nifty timeframe 180. Skipping.
File already exists for Nifty timeframe 240. Skipping.
File already exists for Bank_Nifty timeframe 2. Skipping.
File already exists for Bank_Nifty timeframe 3. Skipping.
File already exists for Bank_Nifty timeframe 5. Skipping.
File already exists for Bank_Nifty timeframe 10. Skipping.
File already exists for Bank_Nifty timeframe 15. Skipping.
File already exists for Bank_Nifty timeframe 20. Skipping.
F

In [13]:
# cell to identify and delete files not matching any instrument_name_timeframe.csv naming
import os

# build a set of valid filenames based on index_mapping keys and the timeframes
valid_filenames = set()
for instr_name in index_mapping.keys():
    for tf in history_config["timeframes"]:
        valid_filenames.add(f"{instr_name}_{tf}.csv")

# list all files in the folder, check which ones are not in the valid set, and delete them
for file in os.listdir(folder_path):
    # skip directories
    if os.path.isdir(os.path.join(folder_path, file)):
        continue
    if file not in valid_filenames:
        file_path = os.path.join(folder_path, file)
        os.remove(file_path)
        print(f"Deleted: {file}")

In [14]:
@njit
def label_signals_jit(close_arr, high_arr, low_arr, target_arr, stoploss_arr):
    n = len(close_arr)
    signals = np.zeros(n, dtype=np.float32)
    entry_prices = np.zeros(n, dtype=np.float32)
    exit_prices = np.zeros(n, dtype=np.float32)
    candles_to_profit = np.zeros(n, dtype=np.float32)
    candles_to_loss = np.zeros(n, dtype=np.float32)

    for i in range(n - 1):
        entry_price = close_arr[i]
        target = target_arr[i]
        stop_loss = stoploss_arr[i]

        buy_target_price = entry_price + target
        buy_sl_price = entry_price - stop_loss
        sell_target_price = entry_price - target
        sell_sl_price = entry_price + stop_loss

        signal_found = False
        for offset in range(i + 1, n):
            future_high = high_arr[offset]
            future_low = low_arr[offset]

            triggers = []
            # Check Buy triggers
            if future_high >= buy_target_price:
                triggers.append((1, offset - i))  # buy target
            if future_low <= buy_sl_price:
                triggers.append((2, offset - i))  # buy stoploss
            # Check Sell triggers
            if future_low <= sell_target_price:
                triggers.append((3, offset - i))  # sell target
            if future_high >= sell_sl_price:
                triggers.append((4, offset - i))  # sell stoploss

            # If any triggers fired, pick the first based on priority (1,2,3,4)
            if triggers:
                triggers.sort(key=lambda x: x[0])
                first_trigger, candle_count = triggers[0]

                signals[i] = float(first_trigger)
                entry_prices[i] = entry_price

                if first_trigger == 1:  # buy target
                    exit_prices[i] = buy_target_price
                    candles_to_profit[i] = float(candle_count)
                elif first_trigger == 2:  # buy stoploss
                    exit_prices[i] = buy_sl_price
                    candles_to_loss[i] = float(candle_count)
                elif first_trigger == 3:  # sell target
                    exit_prices[i] = sell_target_price
                    candles_to_profit[i] = float(candle_count)
                elif first_trigger == 4:  # sell stoploss
                    exit_prices[i] = sell_sl_price
                    candles_to_loss[i] = float(candle_count)

                signal_found = True
                break

        if not signal_found:
            signals[i] = 0
            entry_prices[i] = entry_price

    return signals, entry_prices, exit_prices, candles_to_profit, candles_to_loss


class FullFeaturePipeline:
    def __init__(self, df: pd.DataFrame):
        # df must have columns: [datetime, open, high, low, close, (volume optional)]
        self.df = df.copy()

    def preprocess_datetime(self):
        # Convert Unix timestamp to local time (Asia/Kolkata)
        ist = timezone('Asia/Kolkata')
        self.df['datetime'] = (
            pd.to_datetime(self.df['datetime'], unit='s')
            .dt.tz_localize('UTC')
            .dt.tz_convert(ist)
            .dt.tz_localize(None)
        )

        # Check for duplicates or missing
        if self.df['datetime'].duplicated().any() or self.df['datetime'].isnull().any():
            raise ValueError("The 'datetime' column contains duplicates or missing values.")

        # Sort and set as index
        self.df.sort_values('datetime', inplace=True)
        self.df.set_index('datetime', inplace=True)
        return self

    def clean_data(self):
        #if 'volume' in self.df.columns:
        #   vol_condition = self.df['volume'].isnull() | (self.df['volume'] <= 0)
        #    if vol_condition.any():
        #        self.df.drop('volume', axis=1, inplace=True)

        self.df.drop('volume', axis=1, inplace=True)

        return self

    def add_indicator_features(self):
        # ATR
        self.df['atr_14'] = ta.atr(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            length=14
        ).astype(np.float32).round(2)

        # RSI
        self.df['rsi_14'] = ta.rsi(
            close=self.df['close'],
            length=14
        ).astype(np.float32).round(2)

        # Stoch
        stoch = ta.stoch(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            k=14,
            d=3
        ).astype(np.float32).round(2)
        self.df = self.df.join(stoch)

        # MACD
        macd = ta.macd(
            close=self.df['close'],
            fast=12,
            slow=26,
            signal=9
        ).astype(np.float32).round(2)
        self.df = self.df.join(macd)

        # ADX
        adx = ta.adx(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            length=14
        ).astype(np.float32).round(2)
        self.df = self.df.join(adx[['ADX_14']])
        self.df.rename(columns={'ADX_14': 'adx_14'}, inplace=True)

        # EMAs
        self.df['ema_50'] = ta.ema(self.df['close'], length=50).astype(np.float32).round(2)
        self.df['ema_200'] = ta.ema(self.df['close'], length=200).astype(np.float32).round(2)

        # SMA
        self.df['sma_20'] = ta.sma(self.df['close'], length=20).astype(np.float32).round(2)

        # Keltner Channels
        kc = ta.kc(
            high=self.df['high'],
            low=self.df['low'],
            close=self.df['close'],
            length=20
        ).astype(np.float32).round(2)
        self.df = self.df.join(kc)


        # Price Action Features
        self.df['price_range'] = (self.df['high'] - self.df['low']).astype(np.float32).round(2)
        self.df['body_size'] = (self.df['close'] - self.df['open']).abs().astype(np.float32).round(2)
        self.df['upper_wick'] = (
            self.df['high'] - self.df[['open', 'close']].max(axis=1)
        ).astype(np.float32).round(2)
        self.df['lower_wick'] = (
            self.df[['open', 'close']].min(axis=1) - self.df['low']
        ).astype(np.float32).round(2)

        return self

    def add_time_features(self):
        # Hour, day_of_week, month
        self.df['hour'] = self.df.index.hour.astype(np.int32)
        self.df['day_of_week'] = self.df.index.dayofweek.astype(np.int32)
        self.df['month'] = self.df.index.month.astype(np.int32)
        return self

    def add_adaptive_targets_and_stops(self):
        # ATR-based target and stoploss
        self.df['Target'] = (2.0 * self.df['atr_14']).astype(np.float32).round(2)
        self.df['StopLoss'] = (1.0 * self.df['atr_14']).astype(np.float32).round(2)
        return self

    def label_signals(self):
        from __main__ import label_signals_jit  # or from the same notebook cell, as needed

        close_arr = self.df['close'].values
        high_arr = self.df['high'].values
        low_arr = self.df['low'].values
        target_arr = self.df['Target'].values
        stoploss_arr = self.df['StopLoss'].values

        signals, entry_prices, exit_prices, ctp, ctl = label_signals_jit(
            close_arr,
            high_arr,
            low_arr,
            target_arr,
            stoploss_arr
        )
        self.df['Signal'] = signals.astype(np.float32)
        self.df['Entry Price'] = entry_prices.astype(np.float32).round(2)
        self.df['Exit Price'] = exit_prices.astype(np.float32).round(2)
        self.df['candles_to_profit'] = ctp.astype(np.float32)
        self.df['candles_to_loss'] = ctl.astype(np.float32)

        return self

    def run_pipeline(self):
        (self.preprocess_datetime()
             .clean_data()
             .add_indicator_features()
             .add_time_features()
             .add_adaptive_targets_and_stops()
             .label_signals())
        return self

    def get_processed_df(self):
        # Drop rows with any NaN
        self.df.dropna(axis=0, how='any', inplace=True)

        # Remove Entry/Exit Price columns if you don't need them
        if 'Entry Price' in self.df.columns:
            self.df.drop('Entry Price', axis=1, inplace=True)
        if 'Exit Price' in self.df.columns:
            self.df.drop('Exit Price', axis=1, inplace=True)

        return self.df

In [15]:
if "google.colab" in sys.modules:
    processed_folder_path = history_config["colab_folder_path_processed"]
else:
    processed_folder_path = history_config["local_folder_path_processed"]

if not os.path.exists(processed_folder_path):
    os.makedirs(processed_folder_path)

In [16]:
# We'll store (instrument_name, timeframe, processed_df) tuples in this list
temp_data_list = []

for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        file_path = os.path.join(folder_path, file)

        # Split the filename on the last underscore to accommodate multiple underscores in the instrument name
        instrument_name, timeframe_str = file.replace(".csv", "").rsplit("_", 1)
        timeframe = int(timeframe_str)

        processed_file_name = f"{instrument_name}_{timeframe}_processed.csv"
        processed_file_path = os.path.join(processed_folder_path, processed_file_name)

        # If processed file exists, skip reprocessing
        if os.path.exists(processed_file_path):
            print(f"Processed file already exists for {instrument_name} {timeframe}M. Skipping.")
        else:
            # Process it once, then drop from memory
            raw_df = pd.read_csv(file_path).drop_duplicates(subset='datetime', keep='first')
            pipeline = FullFeaturePipeline(raw_df)
            pipeline.run_pipeline()
            processed_df = pipeline.get_processed_df()

            processed_df.to_csv(processed_file_path, index=True, index_label='datetime')
            print(f"Processed and saved data for {instrument_name} {timeframe}M at {processed_file_path}")

            # Cleanup to free memory
            del raw_df
            del processed_df
            gc.collect()

        # We always add the path so we know it exists (either newly processed or previously)
        temp_data_list.append((instrument_name, timeframe, processed_file_path))

Processed and saved data for Adani_Ports__SEZ_Ltd 240M at processed_data/Adani_Ports__SEZ_Ltd_240_processed.csv
Processed and saved data for Adani_Enterprises_Ltd 60M at processed_data/Adani_Enterprises_Ltd_60_processed.csv
Processed and saved data for Adani_Ports__SEZ_Ltd 180M at processed_data/Adani_Ports__SEZ_Ltd_180_processed.csv
Processed and saved data for Adani_Enterprises_Ltd 240M at processed_data/Adani_Enterprises_Ltd_240_processed.csv
Processed and saved data for Adani_Enterprises_Ltd 10M at processed_data/Adani_Enterprises_Ltd_10_processed.csv
Processed and saved data for Adani_Enterprises_Ltd 45M at processed_data/Adani_Enterprises_Ltd_45_processed.csv
Processed and saved data for Adani_Ports__SEZ_Ltd 20M at processed_data/Adani_Ports__SEZ_Ltd_20_processed.csv
Processed and saved data for Adani_Ports__SEZ_Ltd 15M at processed_data/Adani_Ports__SEZ_Ltd_15_processed.csv
Processed and saved data for Adani_Enterprises_Ltd 30M at processed_data/Adani_Enterprises_Ltd_30_processe

In [17]:
# Sort by timeframe descending, then by instrument name alphabetically
temp_data_list.sort(key=lambda x: (-x[1], x[0]))

instruments_config = []
for i, (instrument_name, timeframe, csv_path) in enumerate(temp_data_list):
    lot_size = index_mapping[instrument_name]["quantity"]
    dynamic_name = f"{instrument_name.upper()}_{timeframe}M"

    instruments_config.append({
        "name": dynamic_name,
        "file_path": csv_path,
        "lot_size": lot_size,
        "transaction_cost": 20.0  # default brokerage
    })

In [18]:
# Now build the main config dictionary
config = {
    "instruments": instruments_config,
    "window_size": 30,
    "unrealized_pnl_weight": 0.1,
    "time_penalty_weight": 0.001,
    "volatility_penalty_weight": 0.05,
    "target_bonus": 0.5,
    "sl_penalty": 0.5,
    "misalignment_penalty": 0.5,
    "missed_opportunity_penalty": 0.05,
    "target_proximity_weight": 0.2,
    "sl_proximity_weight": 0.05,
    "max_step_reward": 5,

    # Trailing Stop Parameters
    "enable_trailing_sl": True,
    "trailing_sl_mode": "factor",   # can be "fixed", "percentage", or "factor"
    "trailing_sl_amount": 20.0,
    "trailing_sl_percent": 0.01,
    "trailing_factor": 0.5,

    "initial_cap_multiplier": 10,   # factor for initial capital
    "normalize_data": True          # whether to normalize data for the agent's observation
}

# Margin constants
MARGIN_FRACTION = 0.05
MARGIN_CALL_THRESHOLD = 0.5

In [19]:
# Print a summary of the dynamically created instruments
print("Final Config Instruments:")
for inst in config["instruments"]:
    print(f"{inst['name']}: lot_size={inst['lot_size']}, brokerage={inst['transaction_cost']}")

Final Config Instruments:
ADANI_ENTERPRISES_LTD_240M: lot_size=10, brokerage=20.0
ADANI_PORTS__SEZ_LTD_240M: lot_size=10, brokerage=20.0
APOLLO_HOSPITALS_ENTERPRISE_LTD_240M: lot_size=10, brokerage=20.0
ASIAN_PAINTS_LTD_240M: lot_size=10, brokerage=20.0
AXIS_BANK_LTD_240M: lot_size=10, brokerage=20.0
BPCL_LTD_240M: lot_size=10, brokerage=20.0
BAJAJ_AUTO_LTD_240M: lot_size=10, brokerage=20.0
BAJAJ_FINANCE_LTD_240M: lot_size=10, brokerage=20.0
BAJAJ_FINSERV_LTD_240M: lot_size=10, brokerage=20.0
BANK_NIFTY_240M: lot_size=30, brokerage=20.0
BANKEX_240M: lot_size=15, brokerage=20.0
BHARAT_ELECTRONICS_LTD_240M: lot_size=10, brokerage=20.0
BHARTI_AIRTEL_LTD_240M: lot_size=10, brokerage=20.0
CIPLA_LTD_240M: lot_size=10, brokerage=20.0
COAL_INDIA_LTD_240M: lot_size=10, brokerage=20.0
DR_REDDYS_LABORATORIES_LTD_240M: lot_size=10, brokerage=20.0
EICHER_MOTORS_LTD_240M: lot_size=10, brokerage=20.0
FINNIFTY_240M: lot_size=40, brokerage=20.0
GRASIM_INDUSTRIES_LTD_240M: lot_size=10, brokerage=20.0
HC

In [20]:
def get_dataframe_by_name(instrument_name, config):
    # instrument_name is something like "BANKNIFTY_5M", "NIFTY50_15M", etc.
    for instrument in config["instruments"]:
        if instrument["name"] == instrument_name:
            return pd.read_csv(instrument["file_path"], parse_dates=['datetime'], index_col='datetime')
    return None


my_df = get_dataframe_by_name("BANK_NIFTY_5M", config)

my_df

,open,high,low,close,atr_14,rsi_14,STOCHk_14_3_3,STOCHd_14_3_3,MACD_12_26_9,MACDh_12_26_9,...,upper_wick,lower_wick,hour,day_of_week,month,Target,StopLoss,Signal,candles_to_profit,candles_to_loss
datetime,,,,,,,,,,,,,,,,,,,,,
2020-09-30 13:20:00,21360.20,21360.70,21340.90,21353.30,49.23,53.68,64.36,69.07,16.04,4.00,...,0.50,12.40,13,2,9,98.46,49.23,2.0,0.0,1.0
2020-09-30 13:25:00,21352.70,21378.80,21284.10,21295.80,52.48,45.53,56.08,62.38,10.83,-0.97,...,26.10,11.70,13,2,9,104.96,52.48,4.0,0.0,1.0
2020-09-30 13:30:00,21294.40,21354.20,21277.80,21346.60,54.19,52.40,51.33,57.25,10.67,-0.90,...,7.60,16.60,13,2,9,108.38,54.19,4.0,0.0,4.0
2020-09-30 13:35:00,21347.90,21351.50,21309.10,21319.10,53.35,48.81,39.66,49.02,8.23,-2.67,...,3.60,10.00,13,2,9,106.70,53.35,4.0,0.0,2.0
2020-09-30 13:40:00,21322.80,21334.50,21294.70,21329.50,52.38,50.20,41.35,44.11,7.06,-3.08,...,5.00,28.10,13,2,9,104.76,52.38,4.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-11-06 15:05:00,52311.15,52315.55,52289.20,52300.80,40.16,44.48,37.44,30.19,-15.79,-5.99,...,4.40,11.60,15,2,11,80.32,40.16,4.0,0.0,2.0
2024-11-06 15:10:00,52303.55,52317.15,52282.90,52294.55,39.74,43.34,39.41,36.14,-15.93,-4.90,...,13.60,11.65,15,2,11,79.48,39.74,4.0,0.0,1.0
2024-11-06 15:15:00,52294.05,52357.65,52293.75,52326.40,41.47,50.32,42.30,39.72,-13.31,-1.83,...,31.25,0.30,15,2,11,82.94,41.47,4.0,0.0,2.0


In [21]:
class Position:
    # Holds all necessary info about the current position (long or short).
    def __init__(self):
        self.reset()

    def reset(self):
        self.status = "flat"             # "flat", "long", or "short"
        self.entry_price = None
        self.sl_points = None            # For initial SL
        self.target_points = None
        self.direction = 0               # +1 for long, -1 for short
        self.quantity = 0
        self.time_in_trade = 0
        self.margin_used = 0.0           # Tracks margin used for the current position

        # Trailing SL related
        self.trailing_sl_price = None    # Absolute trailing SL price
        self.highest_price = None        # For tracking highest price since entry (long)
        self.lowest_price = None         # For tracking lowest price since entry (short)

    def update_unrealized_pnl(self, current_price: float) -> float:
        if self.status == "flat":
            return 0.0
        return (current_price - self.entry_price) * self.direction * self.quantity

    def open(self, entry_price: float, sl_points: float, target_points: float,
             direction: int, quantity: int):
        self.status = "long" if direction == 1 else "short"
        self.entry_price = entry_price
        self.sl_points = sl_points
        self.target_points = target_points
        self.direction = direction
        self.quantity = quantity
        self.time_in_trade = 0

        # Initialize trailing stop to the initial SL directly
        if direction == 1:
            self.trailing_sl_price = entry_price - sl_points
            self.highest_price = entry_price
            self.lowest_price = None
        else:
            self.trailing_sl_price = entry_price + sl_points
            self.highest_price = None
            self.lowest_price = entry_price

In [22]:
class TradingEnv(gym.Env):
    def __init__(self, env_config):
        super(TradingEnv, self).__init__()
        self.config = env_config
        self.total_reward = 0.0

        # Mandatory configuration parameters
        self.window_size = self.config["window_size"]
        self.lot_size = self.config["lot_size"]
        self.transaction_cost = self.config["transaction_cost"]
        self.instrument_name = self.config.get("name", "Unknown")
        self.instrument_id = self.config.get("instrument_id", 0)
        self.normalize_data = self.config.get("normalize_data", False)

        # Data loading using chunking
        self.csv_path = self.config["file_path"]
        self.chunk_size = self.config.get("chunk_size", 10000)
        self.data_iterator = pd.read_csv(
            self.csv_path,
            parse_dates=['datetime'],
            index_col='datetime',
            chunksize=self.chunk_size
        )
        self._load_next_chunk()

        # Initial capital calculation based on the current chunk
        self.max_high = self.buffer_raw['high'].max() if 'high' in self.buffer_raw.columns else 1.0
        multiplier = self.config.get("initial_cap_multiplier", 3)
        self.initial_capital = self.max_high * self.lot_size * MARGIN_FRACTION * multiplier
        self.current_capital = None
        self.current_step = None

        self.position = Position()
        self.trade_log = []

        # Action space: 0 (hold), 1 (long), 2 (short)
        self.action_space = spaces.Discrete(3)

        # Observation space definition remains unchanged
        n_features = self.buffer_obs.shape[1]
        self.observation_space = spaces.Dict({
            "market": spaces.Box(-np.inf, np.inf, shape=(n_features,), dtype=np.float32),
            "position": spaces.Box(-np.inf, np.inf, shape=(4,), dtype=np.float32),
            "history": spaces.Box(-np.inf, np.inf, shape=(self.window_size, n_features), dtype=np.float32),
            "instrument_info": spaces.Box(-np.inf, np.inf, shape=(2,), dtype=np.float32)
        })

        # Reward shaping parameters (unchanged)
        self.unrealized_pnl_weight = self.config.get("unrealized_pnl_weight", 0.1)
        self.time_penalty_weight = self.config.get("time_penalty_weight", 0.02)
        self.volatility_penalty_weight = self.config.get("volatility_penalty_weight", 0.1)
        self.target_bonus = self.config.get("target_bonus", 0.5)
        self.sl_penalty = self.config.get("sl_penalty", 0.5)
        self.misalignment_penalty = self.config.get("misalignment_penalty", 0.5)
        self.missed_opportunity_penalty = self.config.get("missed_opportunity_penalty", 0.05)
        self.target_proximity_weight = self.config.get("target_proximity_weight", 0.2)
        self.sl_proximity_weight = self.config.get("sl_proximity_weight", 0.05)
        self.max_step_reward = self.config.get("max_step_reward", 5.0)

        # Disable trailing stop loss (fixed SL only)
        self.enable_trailing_sl = False

    def _load_next_chunk(self):
        """Loads the next chunk of CSV data and processes it."""
        try:
            chunk = next(self.data_iterator)
        except StopIteration:
            # No more data available; you might want to signal episode termination
            chunk = pd.DataFrame()
        self.total_rows = len(chunk)

        # Create buffers for raw, guidance, and observation
        self.buffer_raw = chunk.drop(columns=['Signal', 'candles_to_profit', 'candles_to_loss'], errors='ignore').copy()
        if 'Signal' in chunk.columns:
            self.buffer_guidance = chunk[['Signal', 'candles_to_profit', 'candles_to_loss']].copy()
        else:
            self.buffer_guidance = pd.DataFrame(index=chunk.index)
        self.buffer_obs = self.buffer_raw.copy()

        # Preprocess data
        self.buffer_raw = self.buffer_raw.ffill().bfill()
        self.buffer_guidance = self.buffer_guidance.ffill().bfill()
        self.buffer_obs = self.buffer_obs.ffill().bfill()

        # Initialize scaler for normalization if needed
        if self.normalize_data:
            self.scaler = StandardScaler()
            self.scaler.fit(self.buffer_obs.values)
            self.buffer_obs = pd.DataFrame(
                self.scaler.transform(self.buffer_obs.values),
                columns=self.buffer_obs.columns,
                index=self.buffer_obs.index
            )
        else:
            self.scaler = None

    def reset(self, seed=None, options=None):
        """Resets the environment to its initial state."""
        super().reset(seed=seed)
        # Reset to the beginning of the current chunk
        self.current_step = self.window_size
        if self.current_step >= self.total_rows:
            self.current_step = max(0, self.total_rows - 1)

        self.position.reset()
        self.current_capital = self.initial_capital
        self.trade_log.clear()
        self.total_reward = 0.0

        obs = self._get_observation()
        return obs, {}

    def step(self, action: int):
        """Executes one time step in the environment."""
        # If we reach the end of the current chunk, attempt to load the next chunk
        if self.current_step >= self.total_rows:
            self._load_next_chunk()
            self.current_step = self.window_size  # Reset step index for the new chunk
            if self.total_rows == 0:
                # No more data to process; terminate the episode
                return self._get_observation(), 0.0, True, False, {}

        current_data = self._get_row(self.buffer_raw, self.current_step)
        guidance_row = self._get_row(self.buffer_guidance, self.current_step)
        reward = 0.0

        # (Rest of your step logic remains the same...)
        if self.position.status == "flat":
            reward += self._apply_guidance_reward(action, guidance_row)

        if self.position.status != "flat":
            if self._check_margin_call(current_data['close']):
                reward += self._close_position(current_data, "Margin Call")

        hit_sl, fill_price = self._check_sl_hit(current_data)
        if hit_sl:
            reward += self._close_position(current_data, "SL Hit", override_exit_price=fill_price)
        else:
            hit_target, target_fill_price = self._check_target_hit(current_data)
            if hit_target:
                reward += self._close_position(current_data, "Target Hit", override_exit_price=target_fill_price)

        self._execute_action(action, current_data)

        if self.position.status != "flat":
            self.position.time_in_trade += 1
            unrealized_pnl = self.position.update_unrealized_pnl(current_data['close'])
            reward += self._calculate_reward(unrealized_pnl, current_data)

        self.current_step += 1
        done = False
        if self.current_capital <= 0:
            done = True
            if self.position.status != "flat":
                reward += self._close_position(current_data, "Force Close")

        self.total_reward += reward
        obs = self._get_observation()
        return obs, reward, done, False, {}

    def _execute_action(self, action: int, current_data: pd.Series):
        """Executes the specified action if conditions are met."""
        price = current_data.get('open', 0.0)
        margin_cost = price * self.lot_size * MARGIN_FRACTION

        if action == 1 and self.position.status == "flat":  # Long
            if self.current_capital >= (margin_cost + self.transaction_cost):
                self._open_position("long", current_data)
        elif action == 2 and self.position.status == "flat":  # Short
            if self.current_capital >= (margin_cost + self.transaction_cost):
                self._open_position("short", current_data)
        # action == 0 (hold) does nothing

    def _open_position(self, direction: str, current_data: pd.Series):
        """Opens a new trading position."""
        entry_price = current_data.get('open', 0.0)
        margin_cost = entry_price * self.lot_size * MARGIN_FRACTION
        self.current_capital -= (margin_cost + self.transaction_cost)
        self.position.margin_used = margin_cost

        sl_val = current_data.get('StopLoss', 0.0)
        tg_val = current_data.get('Target', 0.0)
        direction_int = 1 if direction == "long" else -1

        self.position.open(
            entry_price=entry_price,
            sl_points=sl_val,
            target_points=tg_val,
            direction=direction_int,
            quantity=self.lot_size
        )

        self.trade_log.append({
            "entry_time": current_data.name,
            "position": direction,
            "entry_price": entry_price,
            "exit_time": None,
            "exit_price": None,
            "pnl": None,
            "reason": None,
            "initial_capital": self._format_currency(self.initial_capital),
            "points": None,
            "profit": None,
            "win": None,
            "reward": None,
            "time_in_trade": None
        })

    def _close_position(self, current_data: pd.Series, reason: str, override_exit_price=None) -> float:
        """Closes an active position and updates capital and trade log."""
        if self.position.status == "flat":
            return 0.0

        direction = self.position.direction
        exit_price = override_exit_price if override_exit_price is not None else current_data.get('open', 0.0)
        realized_pnl = (exit_price - self.position.entry_price) * direction * self.lot_size
        margin_return = self.position.margin_used
        self.current_capital += (margin_return + realized_pnl - self.transaction_cost)

        trade_return = realized_pnl / self.initial_capital
        total_reward = trade_return
        points = (exit_price - self.position.entry_price) * direction
        formatted_profit = self._format_currency(realized_pnl)
        win = realized_pnl > 0

        for trade in reversed(self.trade_log):
            if trade["exit_time"] is None:
                trade.update({
                    "exit_time": current_data.name,
                    "exit_price": exit_price,
                    "pnl": realized_pnl,
                    "reason": reason,
                    "time_in_trade": self.position.time_in_trade,
                    "points": round(points, 2),
                    "profit": formatted_profit,
                    "win": win,
                    "reward": total_reward
                })
                break

        self.position.reset()
        return total_reward

    def _check_sl_hit(self, current_data: pd.Series):
        """Checks if the stop loss has been hit."""
        if self.position.status == "flat":
            return False, None

        direction = self.position.direction
        entry_price = self.position.entry_price
        current_open = current_data.get('open', 0.0)
        current_high = current_data.get('high', 0.0)
        current_low = current_data.get('low', 0.0)

        if direction == 1:  # Long
            sl_price = entry_price - self.position.sl_points
            if current_open <= sl_price:
                return True, current_open
            elif current_low <= sl_price:
                return True, sl_price
        else:  # Short
            sl_price = entry_price + self.position.sl_points
            if current_open >= sl_price:
                return True, current_open
            elif current_high >= sl_price:
                return True, sl_price
        return False, None

    def _check_target_hit(self, current_data: pd.Series):
        """Checks if the target price has been hit."""
        if self.position.status == "flat":
            return False, None

        direction = self.position.direction
        entry_price = self.position.entry_price
        target_points = self.position.target_points
        current_open = current_data.get('open', 0.0)
        current_high = current_data.get('high', 0.0)
        current_low = current_data.get('low', 0.0)

        if direction == 1:  # Long
            target_price = entry_price + target_points
            if current_open >= target_price:
                return True, current_open
            elif current_high >= target_price:
                return True, target_price
        else:  # Short
            target_price = entry_price - target_points
            if current_open <= target_price:
                return True, current_open
            elif current_low <= target_price:
                return True, target_price
        return False, None

    def _check_margin_call(self, current_price: float) -> bool:
        """Checks if a margin call is triggered."""
        if self.position.status == "flat":
            return False
        maintenance_margin = self.position.margin_used * MARGIN_CALL_THRESHOLD
        unrealized_pnl = self.position.update_unrealized_pnl(current_price)
        equity = self.current_capital + unrealized_pnl
        return equity < maintenance_margin

    def _apply_guidance_reward(self, action, guidance_row):
        """Applies reward based on guidance signals when flat."""
        if guidance_row.empty:
            return 0.0
        signal = guidance_row.get('Signal', 0)
        candles_to_profit = max(guidance_row.get('candles_to_profit', 1.0), 1.0)

        if signal == 1 and action == 1:  # Long signal
            return self.target_bonus / candles_to_profit
        elif signal == 3 and action == 2:  # Short signal
            return self.target_bonus / candles_to_profit
        elif signal in [1, 3] and action != (1 if signal == 1 else 2):
            return -self.misalignment_penalty
        return 0.0

    def _calculate_reward(self, unrealized_pnl: float, current_data: pd.Series) -> float:
        """Calculates step reward for an open position."""
        risk = self.position.sl_points * self.lot_size
        risk_adj_pnl = (unrealized_pnl / risk) * self.unrealized_pnl_weight if risk > 0 else 0.0
        time_penalty = self.time_penalty_weight * (self.position.time_in_trade / 100.0)

        hi, lo, op = current_data.get('high', 0.0), current_data.get('low', 0.0), current_data.get('open', 1.0)
        volatility_penalty = self.volatility_penalty_weight * ((hi - lo) / max(abs(op), 1e-6))

        if self.position.direction == 1:
            sl_price = self.position.entry_price - self.position.sl_points
            sl_distance = max(0, current_data.get('close', 0.0) - sl_price)
            sl_range = self.position.sl_points
        else:
            sl_price = self.position.entry_price + self.position.sl_points
            sl_distance = max(0, sl_price - current_data.get('close', 0.0))
            sl_range = self.position.sl_points
        sl_prox_ratio = min(sl_distance / sl_range, 1.0) if sl_range > 1e-6 else 1.0
        sl_proximity_penalty = self.sl_proximity_weight * (1.0 - sl_prox_ratio)

        target_proximity_bonus = 0.0
        if self.position.target_points > 0:
            target_price = self.position.entry_price + (self.position.direction * self.position.target_points)
            target_range = abs(self.position.target_points)
            close = current_data.get('close', 0.0)
            distance_to_target = max(0, (target_price - close) * self.position.direction)
            if (self.position.direction == 1 and current_data.get('high', 0.0) >= target_price) or \
               (self.position.direction == -1 and current_data.get('low', 0.0) <= target_price):
                target_proximity_bonus -= self.missed_opportunity_penalty
            if target_range > 1e-6:
                target_proximity_bonus += (1.0 - (distance_to_target / target_range)) * self.target_proximity_weight

        total_reward = risk_adj_pnl + target_proximity_bonus - time_penalty - volatility_penalty - sl_proximity_penalty
        return float(np.clip(total_reward, -self.max_step_reward, self.max_step_reward))

    def _get_observation(self) -> dict:
        """Returns the current observation."""
        idx = self.current_step
        row_obs = self._get_row(self.buffer_obs, idx).values.astype(np.float32)
        current_close = self._get_row(self.buffer_raw, idx)['close']
        unrealized_pnl = self.position.update_unrealized_pnl(current_close)
        position_context = np.array([
            1.0 if self.position.status == "long" else 0.0,
            1.0 if self.position.status == "short" else 0.0,
            unrealized_pnl / self.initial_capital,
            self.position.time_in_trade / 100.0
        ], dtype=np.float32)

        history_data = np.array([
            self._get_row(self.buffer_obs, i).values.astype(np.float32) if 0 <= i < self.total_rows
            else np.zeros_like(row_obs)
            for i in range(idx - self.window_size, idx)
        ], dtype=np.float32)

        instrument_info = np.array([self.lot_size / 1000.0, self.transaction_cost / 100.0], dtype=np.float32)
        return {
            "market": row_obs,
            "position": position_context,
            "history": history_data,
            "instrument_info": instrument_info
        }

    def _get_row(self, df: pd.DataFrame, index: int) -> pd.Series:
        """Retrieves a row from a DataFrame by index, handling out-of-bounds cases."""
        if index < 0 or index >= len(df) or df.empty:
            return pd.Series(np.zeros(df.shape[1] if df.shape[1] else 0, dtype=np.float32), index=df.columns if not df.empty else [])
        return df.iloc[index]

    def get_metrics(self) -> dict:
        """Returns performance metrics based on trade log."""
        if not self.trade_log:
            return {"instrument": self.instrument_name, "message": "No trades taken"}

        closed_trades = [t for t in self.trade_log if t['exit_time'] is not None]
        total_trades = len(closed_trades)
        if total_trades == 0:
            return {"instrument": self.instrument_name, "message": "No trades closed yet"}

        winning_trades = [t for t in closed_trades if t['pnl'] > 0]
        losing_trades = [t for t in closed_trades if t['pnl'] <= 0]
        win_rate = (len(winning_trades) / total_trades) * 100.0
        sum_winning = sum(t['pnl'] for t in winning_trades)
        sum_losing = sum(t['pnl'] for t in losing_trades)
        profit_factor = sum_winning / abs(sum_losing) if sum_losing != 0 else float('inf')

        return {
            "instrument": self.instrument_name,
            "initial_capital": self._format_currency(self.initial_capital),
            "final_capital": self._format_currency(self.current_capital),
            "total_trades": total_trades,
            "win_rate": f"{round(win_rate, 2)}%",
            "total_reward": round(self.total_reward, 2),
            "max_drawdown": self._compute_max_drawdown(),
            "profit_factor": profit_factor
        }

    def _compute_max_drawdown(self) -> float:
        """Calculates the maximum drawdown from the equity curve."""
        equity_curve = [self.initial_capital]
        for t in self.trade_log:
            if t['pnl'] is not None:
                equity_curve.append(equity_curve[-1] + t['pnl'])
        equity_curve = np.array(equity_curve, dtype=np.float32)
        if len(equity_curve) < 2:
            return 0.0
        peak = np.maximum.accumulate(equity_curve)
        dd = (peak - equity_curve) / peak
        return float(dd.max())

    def _format_currency(self, value: float) -> str:
        """Formats a value into a currency string with appropriate units."""
        abs_value = abs(value)
        if abs_value >= 1e7:
            return f"₹{value / 1e7:.2f}Cr"
        elif abs_value >= 1e5:
            return f"₹{value / 1e5:.2f}L"
        elif abs_value >= 1e3:
            return f"₹{value / 1e3:.2f}K"
        else:
            return f"₹{value:.2f}"

In [23]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
from tqdm import tqdm

# Enable cuDNN benchmarking for improved performance with fixed input sizes.
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

# -------------------------------
# Mixture-of-Experts (MoE) Module
# -------------------------------
class MoE(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_experts=4):
        super(MoE, self).__init__()
        self.num_experts = num_experts
        # Define a list of expert networks.
        self.experts = nn.ModuleList([
            nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, output_dim)
            ) for _ in range(num_experts)
        ])
        # Gating network: computes a weight for each expert.
        self.gate = nn.Linear(input_dim, num_experts)

    def forward(self, x):
        # x shape: [batch, seq, input_dim]
        gate_logits = self.gate(x)  # shape: [batch, seq, num_experts]
        gate_weights = torch.softmax(gate_logits, dim=-1)  # [batch, seq, num_experts]

        expert_outputs = []
        for expert in self.experts:
            expert_out = expert(x)  # shape: [batch, seq, output_dim]
            expert_outputs.append(expert_out.unsqueeze(-1))  # add expert dimension

        # Concatenate expert outputs: shape becomes [batch, seq, output_dim, num_experts]
        expert_outputs = torch.cat(expert_outputs, dim=-1)
        # Align gate weights: [batch, seq, 1, num_experts]
        gate_weights = gate_weights.unsqueeze(2)
        # Weighted sum over experts.
        output = torch.sum(expert_outputs * gate_weights, dim=-1)  # [batch, seq, output_dim]
        return output

# --------------------------------------------
# Advanced Transformer Layer with Self-Evaluation
# --------------------------------------------
class AdvancedTransformerLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, num_experts=4):
        super(AdvancedTransformerLayer, self).__init__()
        # Multihead latent self-attention layer.
        self.self_attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        # MoE-based feed-forward module.
        self.moe = MoE(d_model, d_ff, d_model, num_experts)
        self.norm2 = nn.LayerNorm(d_model)
        # Self-evaluation block: a small feed-forward network to refine decisions.
        self.self_eval = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model)
        )
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, x):
        # Multihead self-attention with residual connection.
        attn_out, _ = self.self_attn(x, x, x)
        x = self.norm1(x + attn_out)
        # MoE-based feed-forward with residual connection.
        moe_out = self.moe(x)
        x = self.norm2(x + moe_out)
        # Self-evaluation (refinement) module with residual connection.
        eval_out = self.self_eval(x)
        x = self.norm3(x + eval_out)
        return x

# --------------------------------------------
# Deep Transformer Policy Model (10 layers)
# --------------------------------------------
class DeepTransformerPolicy(nn.Module):
    def __init__(self, history_len, n_features, d_model=64, n_heads=4, num_layers=10, d_ff=None, num_experts=4):
        super(DeepTransformerPolicy, self).__init__()
        self.history_len = history_len
        self.n_features = n_features
        if d_ff is None:
            d_ff = d_model * 4  # Default feed-forward dimension.

        # Embedding layers for various observation components.
        self.market_embed = nn.Linear(n_features, d_model)
        self.position_embed = nn.Linear(4, d_model)
        self.instrument_embed = nn.Linear(2, d_model)
        self.history_embed = nn.Linear(n_features, d_model)

        # Stack of advanced transformer layers.
        self.layers = nn.ModuleList([
            AdvancedTransformerLayer(d_model, n_heads, d_ff, num_experts) for _ in range(num_layers)
        ])

        # Final output head to produce action logits (3 actions: hold, long, short).
        self.fc_out = nn.Linear(d_model, 3)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, obs):
        # Use the device of the model parameters.
        device = next(self.parameters()).device

        # Convert observation components to tensors on the correct device.
        market = torch.tensor(obs["market"], dtype=torch.float32, device=device).unsqueeze(0)      # [1, n_features]
        position = torch.tensor(obs["position"], dtype=torch.float32, device=device).unsqueeze(0)  # [1, 4]
        history = torch.tensor(obs["history"], dtype=torch.float32, device=device).unsqueeze(0)    # [1, window_size, n_features]
        instrument = torch.tensor(obs["instrument_info"], dtype=torch.float32, device=device).unsqueeze(0)  # [1, 2]

        # Embed each component.
        market_emb = self.market_embed(market).unsqueeze(1)         # [1, 1, d_model]
        position_emb = self.position_embed(position).unsqueeze(1)     # [1, 1, d_model]
        instrument_emb = self.instrument_embed(instrument).unsqueeze(1) # [1, 1, d_model]

        batch_size, window_size, _ = history.shape
        history_emb = self.history_embed(history.view(batch_size * window_size, self.n_features))
        history_emb = history_emb.view(batch_size, window_size, -1)    # [1, window_size, d_model]

        # Concatenate all embedded components along the sequence dimension.
        x = torch.cat([market_emb, position_emb, instrument_emb, history_emb], dim=1)  # [1, seq_len, d_model]

        # Process through each advanced transformer layer.
        for layer in self.layers:
            x = layer(x)

        # Aggregate the sequence (e.g., via mean pooling) to obtain a latent representation.
        latent = x.mean(dim=1)  # [1, d_model]
        policy_out = self.fc_out(latent)  # [1, 3]
        action_probs = self.softmax(policy_out)
        return action_probs

# ---------------------------
# Training Function with Checkpointing and AMP
# ---------------------------
def train_grpo(policy, envs, chunk_size=10, epochs=10, lr=1e-4, checkpoint_path="best_checkpoint.pth"):
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    policy.to(device)

    # Use AMP if on CUDA.
    use_amp = device.type == 'cuda'
    scaler = torch.amp.GradScaler(device_type='cuda') if use_amp else None

    best_avg_reward = -float("inf")

    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}/{epochs}")
        epoch_rewards = []
        for i in range(0, len(envs), chunk_size):
            chunk_envs = envs[i:i + chunk_size]
            total_reward = 0

            for env in tqdm(chunk_envs, desc=f"Training chunk {i//chunk_size + 1}"):
                obs, _ = env.reset()
                done = False
                episode_reward = 0
                log_probs = []
                rewards = []

                while not done:
                    # Forward pass under AMP autocast.
                    with torch.amp.autocast(device_type='cuda', enabled=use_amp):
                        action_probs = policy(obs).squeeze(0)
                    dist = Categorical(action_probs)
                    action = dist.sample()
                    log_prob = dist.log_prob(action)

                    next_obs, reward, done, _, _ = env.step(action.item())
                    log_probs.append(log_prob)
                    rewards.append(reward)
                    obs = next_obs
                    episode_reward += reward

                # Compute discounted returns.
                returns = []
                R = 0
                for r in reversed(rewards):
                    R = r + 0.99 * R
                    returns.insert(0, R)
                returns = torch.tensor(returns, dtype=torch.float32, device=device)
                returns = (returns - returns.mean()) / (returns.std() + 1e-8)

                # Compute policy loss.
                policy_loss = 0
                for log_prob, R in zip(log_probs, returns):
                    policy_loss -= log_prob * R
                policy_loss /= len(rewards)

                optimizer.zero_grad()
                # Backward pass with AMP scaling if enabled.
                if use_amp:
                    scaler.scale(policy_loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    policy_loss.backward()
                    optimizer.step()

                total_reward += episode_reward

            avg_reward = total_reward / len(chunk_envs)
            epoch_rewards.append(avg_reward)
            print(f"Chunk {i//chunk_size + 1} Avg Reward: {avg_reward:.2f}")

            # Optional: print instrument-specific metrics.
            print("\nInstrument Metrics for Chunk", i//chunk_size + 1)
            for env in chunk_envs:
                metrics = env.get_metrics()
                print(f"\nInstrument: {env.instrument_name}")  # Assumes `instrument_name` exists.
                for key, value in metrics.items():
                    print(f"{key}: {value}")

        # Compute overall average reward for the epoch.
        epoch_avg_reward = sum(epoch_rewards) / len(epoch_rewards)
        print(f"Epoch {epoch + 1} Average Reward: {epoch_avg_reward:.2f}")

        # Save the best checkpoint.
        if epoch_avg_reward > best_avg_reward:
            best_avg_reward = epoch_avg_reward
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': policy.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_avg_reward': best_avg_reward,
            }, checkpoint_path)
            print(f"New best model saved with average reward: {best_avg_reward:.2f}")

    return policy

# ---------------------------
# Model Configuration and Environment Setup
# ---------------------------
# Assume `config` and `TradingEnv` are already defined elsewhere.
model_config = {
    "history_len": config["window_size"],
    "n_features": None,  # To be set after inspecting an observation.
    "d_model": 64,
    "n_heads": 4,
    "num_layers": 10,   # Using 10 layers as requested.
    "d_ff": 256,        # Typically 4x d_model.
    "num_experts": 4,
}

# Set up environments.
envs = []
for inst in config["instruments"]:
    env_config = config.copy()
    env_config.update(inst)
    env_config.pop("instruments")
    envs.append(TradingEnv(env_config))

# Initialize one environment to determine observation shape.
sample_env = envs[0]
obs, _ = sample_env.reset()

# Determine n_features from the market observation and update model_config.
n_features = obs["market"].shape[-1]
model_config["n_features"] = n_features

# Create the deep transformer policy model using the centralized configuration.
policy = DeepTransformerPolicy(**model_config)

# Move the model to the appropriate device.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy.to(device)

# Wrap with DataParallel if multiple GPUs are available.
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs for DataParallel")
    policy = nn.DataParallel(policy)

# Check for an existing checkpoint.
checkpoint_path = "best_checkpoint.pth"
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    policy.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded checkpoint from epoch {checkpoint['epoch']} with best average reward {checkpoint['best_avg_reward']:.2f}")

# Train the policy (this will resume from the loaded checkpoint if available).
trained_policy = train_grpo(policy, envs, chunk_size=10, epochs=5, checkpoint_path=checkpoint_path)

<ipython-input-23-873d63fc7e1c>:148: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if use_amp else None


Epoch 1/5


Training chunk 1:   0%|          | 0/10 [00:00<?, ?it/s]<ipython-input-23-873d63fc7e1c>:168: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):
Training chunk 1:   0%|          | 0/10 [01:17<?, ?it/s]


ValueError: Found array with 0 sample(s) (shape=(0, 0)) while a minimum of 1 is required by StandardScaler.

In [ ]:
print("\nEnvironment Metrics:")
for env in envs:
    metrics = env.get_metrics()
    print("\nInstrument Metrics:")
    for key, value in metrics.items():
        print(f"{key}: {value}")